### How Persistence helps in Fault Tolerance ?

In [4]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver 
from pydantic import BaseModel
import time

# defining state
class SimpleState(BaseModel):
    name: str = ""
    step1: str = ""
    step2: str = ""
    step3: str = ""


In [5]:
# define node functions
def node_1_func(state: SimpleState) -> dict:
    print("step 1 executed done")
    return {"step1": "done"}

# lets create a fault at node2 set 30 second delay 

def node_2_func(state: SimpleState) -> dict:
    print("step 2 hanging... wait we are trying...") # simulating the fault scenarios
    time.sleep(30)
    return {"step2":"done"}

def node_3_func(state: SimpleState) -> dict:
    print("step 3 executed done")
    return {"step3": "done"}

In [6]:


graph = StateGraph(SimpleState)

# add nodes 
graph.add_node('node1',node_1_func)
graph.add_node('node2',node_2_func)
graph.add_node('node3',node_3_func)

# add edges
graph.add_edge(START,'node1')
graph.add_edge('node1','node2')
graph.add_edge('node2','node3')
graph.add_edge('node3',END)

checkpoint = InMemorySaver()

workflow = graph.compile(checkpoint)



In [7]:
config = {
    "configurable":{
        "thread_id":"1"
    }
}

initial_state = {"name":"john doe"}

try:
    print("Executing please wait....")
    workflow.invoke(initial_state,config)
except KeyboardInterrupt:
    print("Kernel interupted manually X X X ")

Executing please wait....
step 1 executed done
step 2 hanging... wait we are trying...
Kernel interupted manually X X X 


In [ ]:
final_state

{'name': 'John Doe', 'email': 'john@gmail.com', 'age': 30}

In [8]:
# final state after execution complete

workflow.get_state(config)

StateSnapshot(values={'name': 'john doe', 'step1': 'done'}, next=('node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c259-661d-8001-42717e9e059c'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-01-15T12:11:45.301455+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c254-680b-8000-b102904d7bc5'}}, tasks=(PregelTask(id='2f82261a-0e52-6d5e-a25e-42a4f8b82c92', name='node2', path=('__pregel_pull', 'node2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [9]:
# each checkpoint/node state entry/log
# and we know get_state_history is iterator type so we have to list to get logs
list(workflow.get_state_history(config))

[StateSnapshot(values={'name': 'john doe', 'step1': 'done'}, next=('node2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c259-661d-8001-42717e9e059c'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-01-15T12:11:45.301455+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c254-680b-8000-b102904d7bc5'}}, tasks=(PregelTask(id='2f82261a-0e52-6d5e-a25e-42a4f8b82c92', name='node2', path=('__pregel_pull', 'node2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'name': 'john doe'}, next=('node1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c254-680b-8000-b102904d7bc5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-01-15T12:11:45.299457+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c252

- so here, i am going to start the workflow after the state interrupted 

In [ ]:
workflow.invoke(None,config) # this code is used to start the code where it interrupted 

step 2 hanging... wait we are trying...
step 3 executed done


{'name': 'john doe', 'step1': 'done', 'step2': 'done', 'step3': 'done'}

In [ ]:
list(workflow.get_state_history(config))

# final workflow completed 

[StateSnapshot(values={'name': 'john doe', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20bb-e7d9-6d79-8003-fd9e91ccbdd3'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-15T12:14:30.295077+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20bb-e7d7-66b9-8002-12f1bf4d345d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'name': 'john doe', 'step1': 'done', 'step2': 'done'}, next=('node3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20bb-e7d7-66b9-8002-12f1bf4d345d'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-15T12:14:30.294085+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0f20b5-c259-661d-8001-42717e9e059c'}}, tasks=(PregelTask(id='01724b94-ec57-c3c9-075b-656f9877591c', name